Team Members: Saif Eldeen Ali Ahmed Atwaa

Project 5 — Clinic Management
Main Classes: Patient, Doctor, Appointment, MedicalRecord, Payment, **bold text**

---


Features:
• Register patients/doctors

---


• Book/cancel appointments

---


• Search patients

---


• View schedule

---


• Record visits

---


• Track payments

---


Business Rules:
• A doctor cannot have two appointments at the same time

---


• An appointment must belong to an existing patient.

---


• Appointment status: Scheduled, Completed, Cancelled

In [6]:
import json #To import the library that deals with json files (built in library).
import os #importing the operating system library to allow google colab to deal with the operating system like to get some kind of path in the memory or sth.
from abc import ABC, abstractmethod # It allows us to make an abstract method like person which is used below.
from datetime import datetime #Important to use this library allows us to use exact timestamps in our project and this is crucial to deal with schedules like it used below.
from typing import List, Dict, Any, Optional # This one doesnot affect the code logically to much but it enables you to type cast like: str,dict and int just to make the code cleaner.
import ipywidgets as widgets #Library designed mainly for colab and jupyter notebooks. It creates interactive graphical user interface (GUI) elements like buttons, text input boxes, and tab layouts directly inside a web browser.
from IPython.display import display, clear_output #Works with ipywidgets just to render and display tabs, While clear_output is for wiping old text and remove errors and updates info clearly.
# 1. FILE MANAGER (JSON Persistence)
# just to ensure that everything is going right in terms of file managemnt
# Note that it is independent from the main logic of the project just to make it clear and separate to E.Abu Obida when you read it
class FileManager:
    """Handles loading and saving all clinic data to a JSON file."""
    FILE_NAME = "clinic_data.json"#Class-level variable just to define every data part name to be stored in colab virual drive.

    @staticmethod #To make it for the class it self not a specific object or instance.
    def save_data(data: Dict[str, List[Dict[str, Any]]]) -> None: #Just to specify that the entered data will be from a dictionary datatype,None---> to ensure that it doesnot return somthing just it saves the data.
        try:                                                    #just for error handling
            with open(FileManager.FILE_NAME, "w") as f:          #'w' obviously this is for the mode of writing in the file we need to write in.
                json.dump(data, f, indent=4)                     #To take the complex dict and converts it into a readable material in the json file, used AI in that part; cuz I did't use to deal with json before.
        except Exception as e:                                  #Also for error handling in order to print a friendly massage if there is any violation that causes an error in the program.
            print(f"Error saving data: {e}")

    @staticmethod # Another staticmethod just to load that data to present it when we need it
    def load_data() -> Dict[str, List[Dict[str, Any]]]: #just to make a list of dictionaries when it gives me the obtained data
        if not os.path.exists(FileManager.FILE_NAME): # checks if the file exists in my google colab data
            return {"patients": [], "doctors": [], "appointments": [], "records": [], "payments": []} #if it doesnot exist it gives me dictionary of lists peacefully instead of giviing me an error
        try:
            with open(FileManager.FILE_NAME, "r") as f: #'r' for read mode , f is like a varaible just to save the connection between the block of the code and the file on the hard drive.
                return json.load(f)
        except Exception as e: #this for error handling to print a statement instead of giving an error just to make the user aware of entering suffcient data only.
            print(f"Error loading data: {e}")
            return {"patients": [], "doctors": [], "appointments": [], "records": [], "payments": []}

# 2. ABSTRACT BASE CLASS & INHERITANCE
class Person(ABC): #this is an abstract base class which has properties mutual between patient and doctor classes so they will be able to inherit from that class
    """Abstract Base Class representing a human entity in the clinic system."""
    def __init__(self, name: str, phone: str):
        self._name = name
        self._phone = phone

    @property #getter to get the name when we call it
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        if not value.strip(): #.strip() to make it with no spaces between characters
            raise ValueError("Name cannot be empty.") #To make the user aware of entering name before any other step
        self._name = value #it stores the name of the patient as a object or instance

    @property                                   #same as name
    def phone(self) -> str:
        return self._phone

    @phone.setter
    def phone(self, value: str) -> None:
        if not value.strip():
            raise ValueError("Phone cannot be empty.")
        self._phone = value

    @abstractmethod #method just we can get all details in a string data type to display it
    def get_details(self) -> str:
        pass


class Patient(Person): #This part is for inhertance just to patient as a class to inherit from the person (parent class).
    """Patient entity inheriting from Person."""
    def __init__(self, patient_id: str, name: str, phone: str, age: int): #Just to take that kinds of data from the user
        super().__init__(name, phone) # Here is a call to the parent class in order to use name and phone to make another objects in the class of patient
        self._patient_id = patient_id #Just to make every patient unique from the other patients so he will have an unique id from others
        self._age = (age) # to enter age

    @property
    def patient_id(self) -> str: #Getter just to return the unique id of the patient.
        return self._patient_id

    @property
    def age(self) -> int: #Same but for age.
        return self._age

    @age.setter
    def age(self, value: int) -> None:
        if int(value) <= 0: #Here to ensure that it cannot be a negative age or even a zero.
            raise ValueError("Age must be greater than zero.")
        self._age = int(value)

    def get_details(self) -> str:
        return f"Patient [ID: {self._patient_id}] - Name: {self._name}, Age: {self._age}, Phone: {self._phone}" #That's how we will present patient's data when he enters it.

    def to_dict(self) -> Dict[str, Any]:
        return {"patient_id": self._patient_id, "name": self._name, "phone": self._phone, "age": self._age}#This is how it will be saved in the json file


class Doctor(Person): # The class of doctor inherited from person.
    """Doctor entity inheriting from Person.""" #Doc string.
    def __init__(self, doctor_id: str, name: str, phone: str, specialization: str): #The same as patient class.
        super().__init__(name, phone)
        self._doctor_id = doctor_id
        self._specialization = specialization

    @property #property getter
    def doctor_id(self) -> str:
        return self._doctor_id

    @property
    def specialization(self) -> str:
        return self._specialization

    def get_details(self) -> str:
        return f"Dr. {self._name} ({self._specialization}) - ID: {self._doctor_id}" #How it will be printed

    def to_dict(self) -> Dict[str, Any]:
        return {"doctor_id": self._doctor_id, "name": self._name, "phone": self._phone, "specialization": self._specialization} #How it will be stored in the json file.
# 3. ADDITIONAL CORE CLASSES (APPOINTMENT ,MEDICAL RECORD AND PAYMENT CLASSES)
class Appointment:
    """Manages appointments linking Patients and Doctors with status control."""
    def __init__(self, app_id: str, patient_id: str, doctor_id: str, date_time: str, status: str = "Scheduled"):
        self.__app_id = app_id #Appointment ID
        self.__patient_id = patient_id # The unique patient ID.
        self.__doctor_id = doctor_id # Id of the doctor who the patient has the appointment with.
        self.__date_time = date_time #To store date and time.
        self.__status = status  # Scheduled, Completed, Cancelled.

    @property #A group of getters that gets every data we want to call.
    def app_id(self) -> str: return self.__app_id
    @property
    def patient_id(self) -> str: return self.__patient_id
    @property
    def doctor_id(self) -> str: return self.__doctor_id
    @property
    def date_time(self) -> str: return self.__date_time
    @property
    def status(self) -> str: return self.__status
    @status.setter
    def status(self, val: str) -> None: #Setter for the status
        if val not in ["Scheduled", "Completed", "Cancelled"]:
            raise ValueError("Invalid appointment status.")
        self.__status = val

    def to_dict(self) -> Dict[str, Any]:
        return {
            "app_id": self.__app_id,
            "patient_id": self.__patient_id,
            "doctor_id": self.__doctor_id,
            "date_time": self.__date_time,
            "status": self.__status
        }


class MedicalRecord:
    """Stores visit diagnoses and medical history notes for patients."""
    def __init__(self, record_id: str, patient_id: str, diagnosis: str, prescription: str):
        self.__record_id = record_id
        self.__patient_id = patient_id
        self.__diagnosis = diagnosis
        self.__prescription = prescription

    @property #getters and encapsulation
    def record_id(self) -> str: return self.__record_id
    @property
    def patient_id(self) -> str: return self.__patient_id
    @property
    def diagnosis(self) -> str: return self.__diagnosis
    @property
    def prescription(self) -> str: return self.__prescription

    def to_dict(self) -> Dict[str, Any]: #As mentioned before this to store it in a dictionary in order to give it to json file.
        return {
            "record_id": self.__record_id,
            "patient_id": self.__patient_id,
            "diagnosis": self.__diagnosis,
            "prescription": self.__prescription
        }


class Payment: #Same as medical record and appointment classes.
    """Tracks billing and payments for clinical services.""" #Doc string.
    def __init__(self, payment_id: str, patient_id: str, amount: float, status: str = "Paid"): #Here we set a defult value for status.
        if float(amount) <= 0: #Payment cannot be negative.
            raise ValueError("Payment amount must be greater than zero.")
        self.__payment_id = payment_id
        self.__patient_id = patient_id
        self.__amount = float(amount)
        self.__status = status

    @property #Getters and encapsulation.
    def payment_id(self) -> str: return self.__payment_id
    @property
    def patient_id(self) -> str: return self.__patient_id
    @property
    def amount(self) -> float: return self.__amount
    @property
    def status(self) -> str: return self.__status

    def to_dict(self) -> Dict[str, Any]: #As the privous dicts.
        return {
            "payment_id": self.__payment_id,
            "patient_id": self.__patient_id,
            "amount": self.__amount,
            "status": self.__status
        }
# 4. CLINIC MANAGER (Business Logic & CRUD)
class ClinicSystem:
    def __init__(self):
        raw = FileManager.load_data() #To load the data when we want it to be displayed or to do anything on it.
        self.patients = [Patient(**p) for p in raw["patients"]] #List comprhension and Unpacking the whole data, we iterate every part just to read the whole dicionary of every patient or any object and the ** for unpacking dicitonary.
        self.doctors = [Doctor(**d) for d in raw["doctors"]]
        self.appointments = [Appointment(**a) for a in raw["appointments"]]
        self.records = [MedicalRecord(**r) for r in raw["records"]]
        self.payments = [Payment(**pay) for pay in raw["payments"]]

    def save(self): #This to save every part of data in an iterable dictionary and makes a packaging for every data part in FileManager.save_data() just to make it safely for the JSON file.
        FileManager.save_data({
            "patients": [p.to_dict() for p in self.patients],
            "doctors": [d.to_dict() for d in self.doctors],
            "appointments": [a.to_dict() for a in self.appointments],
            "records": [r.to_dict() for r in self.records],
            "payments": [p.to_dict() for p in self.payments]
        })
    # There are helper methods to find the desired patient or doctor by their IDs
    # Business Rule: Patient must exist before, like it must be entered before
    def find_patient(self, pid: str) -> Optional[Patient]: #It looks through self.patients to find the first patient whose ID matches pid, I needed AI help to know how to search for a patient in the list.
        return next((p for p in self.patients if p.patient_id == pid), None) # It iterates for the whole list, if it doesnot find it it gives none

    def find_doctor(self, did: str) -> Optional[Doctor]: #Same as find patient
        return next((d for d in self.doctors if d.doctor_id == did), None)

    def add_patient(self, pid, name, phone, age):
        if self.find_patient(pid): # To prevent duplication
            raise ValueError(f"Patient ID {pid} already exists.")
        if not name.strip(): #.strip() to make it with no spaces between characters
            raise ValueError("Name cannot be empty.")
        if int(age) <= 0:
            raise ValueError("Age must be greater than zero.")
        p = Patient(pid, name, phone, int(age))
        self.patients.append(p) # Here if there is no duplication there, It appends the new patient to the list of patients.
        self.save() # And saves it in the data

    def add_doctor(self, did, name, phone, spec): #Same as the previous.
        if self.find_doctor(did):
            raise ValueError(f"Doctor ID {did} already exists.")
        d = Doctor(did, name, phone, spec)
        self.doctors.append(d)
        self.save()

    def add_appointment(self, aid, pid, did, date_time): # A Method to add an appointment or book it.
        if not self.find_patient(pid): # Patient ID must exist that means that the patient exist on the system.
            raise ValueError("Business Rule Violation: Appointment must belong to an existing patient.")
        if not self.find_doctor(did): #You must book with a doctor who exsits
            raise ValueError("Selected doctor does not exist.")

        # Business Rule: A doctor cannot have two appointments at the exact same time
        for app in self.appointments:
            if app.doctor_id == did and app.date_time == date_time and app.status == "Scheduled": #To check that doctor has an appointment at this time and its status is "Scheduled"
                raise ValueError("Business Rule Violation: Doctor already has an appointment at this time slot.")

        a = Appointment(aid, pid, did, date_time, "Scheduled")
        self.appointments.append(a) #If there is Business Rule Violation it will append that appointment to the list of appointments.
        self.save() #And will save it.

    def add_record(self, rid, pid, diagnosis, rx):#To add a medical record to the system
        if not self.find_patient(pid): #To check if the patient exists on the system before.
            raise ValueError("Patient does not exist.")
        r = MedicalRecord(rid, pid, diagnosis, rx)
        self.records.append(r) #It appends it to the list of the medical records.
        self.save() #And saves it.

    def add_payment(self, pay_id, pid, amount):
        if not self.find_patient(pid): #This is to check that the patient is on the system because no one can pay if he is not on the system just to make it ordered well.
            raise ValueError("Patient does not exist.")
        p = Payment(pay_id, pid, float(amount))
        self.payments.append(p) #So he can pay and it will be saved to the system
        self.save()
# 4. GOOGLE COLAB IPYWIDGETS GUI DASHBOARD
# This part I tried hardly to understand it but it was mainly by AI.
def build_gui(): #function to build the gui dashboard
    system = ClinicSystem() #to call the class of clinicsystem in a variable called system

    out_output = widgets.Output() #Storing method of the library in an variable to be called later.

    # Tabs
    tab = widgets.Tab() #Method to add tabs

   # the dashboard
    def refresh_dashboard(_=None): #This makes the whole thing easier instead of writing hard code blocks, it counts how many patients,doctor,appointments..etc) and displays it.
        with out_output:
            clear_output(wait=True)
            print("--- CLINIC MANAGEMENT DASHBOARD ---")
            print(f"Total Patients registered: {len(system.patients)}")
            print(f"Total Doctors on staff: {len(system.doctors)}")
            print(f"Active Appointments: {sum(1 for a in system.appointments if a.status == 'Scheduled')}")
            print(f"Total Medical Records: {len(system.records)}")
            print(f"Total Collected Revenue: ${sum(p.amount for p in system.payments)}")

    btn_refresh = widgets.Button(description="Refresh Dashboard", button_style="info") #.Button is a method to make a clickable button. Here is to refresh dashboard to display new saved information.
    btn_refresh.on_click(refresh_dashboard)

    tab_dash = widgets.VBox([btn_refresh, out_output]) #This line groups refresh button and your output text box together into a single vertical layout column, placing the button directly above the dashboard text display on your screen.

    # Patients tabs -------- This is building a form to register patients and their data.
    p_id = widgets.Text(description="ID:")
    p_name = widgets.Text(description="Name:")
    p_phone = widgets.Text(description="Phone:")
    p_age = widgets.IntText(description="Age:", value=25)
    btn_add_p = widgets.Button(description="Register Patient", button_style="success")
    p_output = widgets.Output()

    def on_add_patient(b): #Just to add or append a patient to the system
        with p_output:
            clear_output()
            try: #For error handling.
                system.add_patient(p_id.value, p_name.value, p_phone.value, p_age.value)
                print(f"Success: Patient {p_name.value} registered!")
            except Exception as e:
                print(f"Error: {e}")
    btn_add_p.on_click(on_add_patient) #On_click tells python that whenever the button is pressed run that function.

    #Search filter for patients
    search_p = widgets.Text(description="Search Name:") #This whenever the user clicks search patient after entering a name in search_p it loops through the system to find the match to it and if it exist is prints get details().
    p_list_out = widgets.Output()
    def on_search_patient(b):
        with p_list_out:
            clear_output()
            query = search_p.value.lower()
            results = [p.get_details() for p in system.patients if query in p.name.lower()]
            print("\n".join(results) if results else "No patients found.")
    btn_search_p = widgets.Button(description="Search")
    btn_search_p.on_click(on_search_patient)

    tab_patients = widgets.VBox([widgets.HTML("<b>Add New Patient</b>"), p_id, p_name, p_phone, p_age, btn_add_p, p_output,
                               widgets.HTML("<hr><b>Search Patients</b>"), search_p, btn_search_p, p_list_out])#Widgets.HTML("<b>xxx") this is for displaying a bold HTML text as a section header
                                        #VBox like it said before it is for grouping all the patient data like the dashbaoard.
    # Doctors tab
    d_id = widgets.Text(description="Doc ID:")#Same as patients in making form and appending doctor to the system.
    d_name = widgets.Text(description="Name:")
    d_phone = widgets.Text(description="Phone:")
    d_spec = widgets.Text(description="Specialty:")
    btn_add_d = widgets.Button(description="Add Doctor", button_style="success")
    d_output = widgets.Output()

    def on_add_doctor(b):
        with d_output:
            clear_output()
            try:
                system.add_doctor(d_id.value, d_name.value, d_phone.value, d_spec.value)
                print(f"Success: Dr. {d_name.value} added!")
            except Exception as e:
                print(f"Error: {e}")
    btn_add_d.on_click(on_add_doctor)
    tab_doctors = widgets.VBox([widgets.HTML("<b>Add Doctor</b>"), d_id, d_name, d_phone, d_spec, btn_add_d, d_output])

    #Appointments #SAME AS PREVIOUS
    a_id = widgets.Text(description="App ID:")
    a_pid = widgets.Text(description="Patient ID:")
    a_did = widgets.Text(description="Doctor ID:")
    a_time = widgets.Text(description="Date/Time:", value="2027-04-02 14:00")
    btn_add_a = widgets.Button(description="Book Appointment", button_style="success")
    a_output = widgets.Output()

    def on_add_app(b):
        with a_output:
            clear_output()
            try:
                system.add_appointment(a_id.value, a_pid.value, a_did.value, a_time.value)
                print("Success: Appointment booked!")
            except Exception as e:
                print(f"Error: {e}")
    btn_add_a.on_click(on_add_app)
    tab_apps = widgets.VBox([widgets.HTML("<b>Book Appointment</b>"), a_id, a_pid, a_did, a_time, btn_add_a, a_output])

    # Records & Payments  #Same as previous
    r_id = widgets.Text(description="Rec ID:")
    r_pid = widgets.Text(description="Patient ID:")
    r_diag = widgets.Text(description="Diagnosis:")
    r_rx = widgets.Text(description="Prescription:")
    btn_add_r = widgets.Button(description="Save Medical Record", button_style="success")
    r_output = widgets.Output()

    def on_add_rec(b):
        with r_output:
            clear_output()
            try:
                system.add_record(r_id.value, r_pid.value, r_diag.value, r_rx.value)
                print("Success: Medical record added!")
            except Exception as e:
                print(f"Error: {e}")
    btn_add_r.on_click(on_add_rec)
    tab_records = widgets.VBox([widgets.HTML("<b>Medical Records</b>"), r_id, r_pid, r_diag, r_rx, btn_add_r, r_output])

    # Payments Tab GUI
    pay_id = widgets.Text(description="Pay ID:")
    pay_pid = widgets.Text(description="Patient ID:")
    pay_amt = widgets.FloatText(description="Amount:", value=100.0)
    btn_add_pay = widgets.Button(description="Record Payment", button_style="success")
    pay_output = widgets.Output()

    def on_add_payment(b):
        with pay_output:
            clear_output()
            try:
                system.add_payment(pay_id.value, pay_pid.value, pay_amt.value)
                print(f"Success: Payment of ${pay_amt.value} recorded!")
            except Exception as e:
                print(f"Error: {e}")
    btn_add_pay.on_click(on_add_payment)
    tab_payments = widgets.VBox([widgets.HTML("<b>Process Payment</b>"), pay_id, pay_pid, pay_amt, btn_add_pay, pay_output])

    # Assemble Tabs
    tab.children = [tab_dash, tab_patients, tab_doctors, tab_apps, tab_records, tab_payments] #To display application Tabs.
    tab.set_title(0, "Dashboard")
    tab.set_title(1, "Patients")
    tab.set_title(2, "Doctors")
    tab.set_title(3, "Appointments")
    tab.set_title(4, "Medical Records")
    tab.set_title(5, "Payments")

    display(tab)
    refresh_dashboard()

#To run the GUI application
build_gui()